### Goal: Prepare data for DBeaver import and subsequent SQL analysis.

### Main Steps:
* **Cleaning:** Handling non-standard formats (replacing delimiters, removing unnecessary characters).
* **Transformation:** Converting a single string with a list of items into a "one row — one item" format, which is essential for accurate SQL analysis.
* **Preparation:** Casting data types to the format required for MySQL (DBeaver) loading.

In [ ]:
import pandas as pd

# Load source data

df = pd.read_csv('receipts.csv')

# Replace commas with dots to handle decimal numbers correctly

df['Описание'] = df['Описание'].str.replace('0,19', '0.19', regex = False)
df['Описание'] = df['Описание'].str.replace('0,2', '0.2', regex = False)
df['Описание'] = df['Описание'].str.replace('0,3', '0.3', regex = False)
df['Описание'] = df['Описание'].str.replace('0,4', '0.4', regex = False)
df['Описание'] = df['Описание'].str.replace('0,5', '0.5', regex = False)

# Create a copy for further processing

df_temp = df.copy()

# Split the string by comma to create a list of items

df_temp['items'] = df['Описание'].str.split(',')

# Flatten the list into individual rows

df_final = df_temp.explode('items')

# Clean whitespace and quotes
df_final['items'] = df_final['items'].str.replace('"', "", regex = False).str.strip()

# Remove empty rows
df_final = df_final.dropna(subset=['items'])

# Split the item string into amount and product name
split_data = df_final['items'].str.split('x', n=1, expand=True)
df_final['item_amount'] = split_data[0]
df_final['item_prod'] = split_data[1]

# Trim whitespace and convert amount to numeric format
df_final['item_amount'] = df_final['item_amount'].str.strip()
df_final['item_prod'] = df_final['item_prod'].str.strip()

df_final['item_amount'] = pd.to_numeric(df_final['item_amount'], errors='coerce')

print(df_final[['item_amount', 'item_prod']].head())

   item_amount      item_prod
0          1.0    Potato cake
1          1.0            Tea
2          1.0  americano 0.2
2          1.0  americano 0.3
2          1.0  Carrot pieces


In [ ]:
# Save the cleaned data to a CSV file for import into DBeaver
cols_to_keep = [
    'Дата', 'Номер чека', 'Продажи', 'Скидки', 'Выручка', 'Налоги', 
    'Итого', 'Себестоимость товаров', 'Валовая прибыль', 
    'item_amount', 'item_prod', 'Касса'
]
df_final_clean = df_final[cols_to_keep]
df_final_clean.to_csv('data_for_dbeaver_v2.csv', index=False, sep=';', encoding='utf-8')


In [ ]:
# calculate the receipt depth and make a heath map
SELECT DAYNAME(created_at), 
HOUR(created_at), 
ROUND(SUM(item_amount) * 1.0 / COUNT(DISTINCT receipt_id), 2) AS depth_chek,
ROUND(SUM(earning), 2) AS earning_per_hour
FROM fox 
GROUP BY DAYNAME(created_at), HOUR(created_at)
ORDER BY 1, 2;

# evaluate order amount per dayhour # traffic

SELECT 
    HOUR(created_at) AS hour_of_day,
    COUNT(DISTINCT CASE WHEN DAYNAME(created_at) = 'Monday' THEN receipt_id END) AS Monday,
    COUNT(DISTINCT CASE WHEN DAYNAME(created_at) = 'Tuesday' THEN receipt_id END) AS Tuesday,
    COUNT(DISTINCT CASE WHEN DAYNAME(created_at) = 'Wednesday' THEN receipt_id END) AS Wednesday,
    COUNT(DISTINCT CASE WHEN DAYNAME(created_at) = 'Thursday' THEN receipt_id END) AS Thursday,
    COUNT(DISTINCT CASE WHEN DAYNAME(created_at) = 'Friday' THEN receipt_id END) AS Friday,
    COUNT(DISTINCT CASE WHEN DAYNAME(created_at) = 'Saturday' THEN receipt_id END) AS Saturday,
    COUNT(DISTINCT CASE WHEN DAYNAME(created_at) = 'Sunday' THEN receipt_id END) AS Sunday
FROM fox
GROUP BY HOUR(created_at)
ORDER BY HOUR(created_at);

# CALCULATE A differ in minutes between orders
WITH lag_steps AS (
SELECT 
DAYNAME(created_at) AS day_name, 
HOUR(created_at) AS hour_day,
  # Считаем разницу только внутри конкретной даты 
TIMESTAMPDIFF(MINUTE, 
LAG(created_at) OVER (PARTITION BY DATE(created_at) ORDER BY created_at), 
created_at
) AS diff_minutes
FROM fox
)
SELECT 
day_name, 
hour_day,
ROUND(AVG(diff_minutes), 2) AS avg_diff
FROM lag_steps 
WHERE diff_minutes IS NOT NULL 
AND diff_minutes < 60
GROUP BY 1, 2
ORDER BY FIELD(day_name, 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'), hour_day;


# calculate amount of orders per hour
SELECT HOUR(created_at) AS time_hour, COUNT(DISTINCT receipt_id) AS order_per_hour, SUM(earning) AS sum_hour
FROM fox f 
GROUP BY HOUR(created_at)
ORDER BY 2 DESC;

# determine total income, total profit per weekday
SELECT WEEKDAY(created_at), SUM(earning), ROUND(SUM(profit), 2)
FROM fox f GROUP BY 1;

# calculate average receipt for each hour per weekday
SELECT DAYNAME(created_at) AS week_day, HOUR(created_at) AS hour_day,
COUNT(DISTINCT receipt_id) AS order_per_hour, ROUND(SUM(earning) / COUNT(DISTINCT receipt_id), 2) AS avg_receipt_value, SUM(earning) AS total_earn
FROM fox 
GROUP BY DAYNAME(created_at), HOUR(created_at)
ORDER BY 3 DESC;

# select items of top earning days 
SELECT item_prod, COUNT(item_prod)
FROM fox 
WHERE 
WEEKDAY(created_at) = 4 AND HOUR(created_at) IN(10, 11, 12)
GROUP BY item_prod
ORDER BY 2 DESC;

# понять с каким количеством заказов имеем дело в час пик
SELECT  COUNT(DISTINCT receipt_id)
FROM fox 
WHERE 
WEEKDAY(created_at) = 4 AND HOUR(created_at) IN(10, 11, 12);


# Monthly trend analysis: Revenue and transaction count
SELECT DATE_FORMAT(created_at, '%Y-%m') AS month, SUM(earning) AS earning_month, COUNT(DISTINCT receipt_id) AS orders_total
FROM fox 
GROUP BY DATE_FORMAT(created_at, '%Y-%m') 
ORDER BY 1;



# calculate a base for metrics for each hour of each day 
CREATE TABLE hour_volat AS
WITH daily_hourly_stats AS (
SELECT 
        DAYNAME(created_at) AS week_day,
        HOUR(created_at) AS hour_day,
        COUNT(DISTINCT receipt_id) AS orders,
        SUM(earning) AS earn
FROM fox
GROUP BY DATE(created_at), DAYNAME(created_at), HOUR(created_at))
# calculate sd, var to evaluate a difference between working hours forexample for all mondays
SELECT 
    week_day,
    hour_day,
    COUNT(orders) AS orders_amount,
    ROUND(STDDEV(orders), 2) AS orders_stddev,
    ROUND(AVG(earn), 2) AS avg_earn_per_hour,
    ROUND(STDDEV(earn), 2) AS earn_stddev,
    ROUND((STDDEV(earn) / AVG(earn)) * 100, 1) AS variation
FROM daily_hourly_stats
GROUP BY week_day, hour_day
HAVING (STDDEV(earn) / AVG(earn)) > 0.5
ORDER BY variation DESC;

# determination of the most volatile clocks within each day
SELECT week_day, hour_day,
orders_amount,
orders_stddev,
avg_earn_per_hour,
earn_stddev,
variation,
ROW_NUMBER() OVER(PARTITION BY week_day ORDER BY variation DESC)
FROM hour_volat;

# identify volatile hours to determine if the issue is staff-related or traffic-driven

SELECT 
    CASE 
        WHEN week_day IN ('Tuesday', 'Wednesday') THEN 'Tuesday/Wednesday_Storm' 
        ELSE 'rest_days' 
    END AS day_group,
    -- Средняя эффективность часа
    ROUND(AVG(avg_earn_per_hour), 2) AS avg_revenue_per_hour,
    -- Масштаб (сумма средних выручек)
    ROUND(SUM(avg_earn_per_hour), 2) AS total_potential_revenue,
    -- Средняя нестабильность (волатильность)
    ROUND(AVG(variation), 1) AS avg_instability_pct2
FROM hour_volat
GROUP BY 1
ORDER BY avg_revenue_per_hour DESC;

# Analyze overall performance trends for Tuesdays and Wednesdays

SELECT DATE(created_at), DAYNAME(created_at), SUM(earning), COUNT(DISTINCT receipt_id)
FROM fox 
GROUP BY DATE(created_at), DAYNAME(created_at)
ORDER BY 2, 1;

SELECT 
    DATE(created_at) AS sale_date,
    DAYNAME(created_at) AS day_name,
    SUM(earning) AS daily_earning,
    -- Средняя выручка по ВСЕМ дням за последние 7 дней (скользящее)
    ROUND(AVG(SUM(earning)) OVER (
        ORDER BY DATE(created_at) 
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ), 2) AS weekly_avg,
    -- Отклонение текущего дня от недельного среднего
    ROUND(((SUM(earning) / AVG(SUM(earning)) OVER (
        ORDER BY DATE(created_at) 
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    )) - 1) * 100, 2) AS deviation_pct
FROM fox 
GROUP BY DATE(created_at), DAYNAME(created_at)
ORDER BY sale_date;

SELECT 
    CASE 
        WHEN variation > 150 THEN 'High Chaos'
        WHEN variation BETWEEN 50 AND 150 THEN 'Medium'
        ELSE 'Stable'
    END AS chaos_level,
    AVG(orders_amount) AS avg_orders,
    AVG(avg_earn_per_hour) / AVG(orders_amount) AS avg_check
FROM hour_volat
GROUP BY 1;